In [ ]:
import re
from urllib.parse import urlparse

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# take the dump of the feed common table from the dau-dashboard as a csv file
df = pd.read_csv(r"/path/to/feed_common.csv")
print(df.shape)

In [ ]:
# first convert the time from UTC to IST
df["inserted_at"] = pd.to_datetime(df["inserted_at"])
df["inserted_at_ist"] = (
    df["inserted_at"].dt.tz_localize("UTC").dt.tz_convert("Asia/Kolkata")
)

In [ ]:
df.tail()

In [ ]:
# Set date range here
df = df[df["inserted_at_ist"].dt.date >= pd.to_datetime("2025-01-01").date()]
df = df[df["inserted_at_ist"].dt.date < pd.to_datetime("2025-04-01").date()]

df = df.reset_index(drop=True)
print(df.shape)

In [ ]:
df_sorted = df.sort_values(by="inserted_at_ist", ascending=True)
# Optionally, reset the index after sorting
df_sorted = df_sorted.reset_index(drop=True)
df_sorted = df_sorted[["id", "inserted_at", "inserted_at_ist"]]

In [ ]:
df_sorted.head()

In [ ]:
df_sorted.tail()

## Plot 1
### Pie chart of media type messages recieved on the tipline

In [ ]:
video_count = df[df["media_type"] == "video"].shape[0]
audio_count = df[df["media_type"] == "audio"].shape[0]
url_count = df[df["media_type"] == "text"].shape[0]
media_count = video_count + audio_count

print(f"media_count: {media_count}")
print(f"audio_count: {audio_count}")
print(f"video_count: {video_count}")
print(f"url_count: {url_count}")

In [ ]:
labels = ["Audio", "Video", "Text"]
counts = [audio_count, video_count, url_count]
total_count = sum(counts)

percentages = [(count / total_count) * 100 for count in counts]

plot1_df = pd.DataFrame(
    {
        "Label": labels,
        "Count": counts,
        "Percentage": [f"{percentage:.2f}%" for percentage in percentages],
    }
)
print(plot1_df)

In [ ]:
colors = ["#93AF7B", "#F0A7C4", "#70ACE8"]
plt.figure(figsize=(4, 4))
plt.pie(
    plot1_df["Count"],
    labels=plot1_df["Label"],
    colors=colors,
    wedgeprops={"edgecolor": "black"},
)
plt.title("Distribution of Media Types")
plt.axis("equal")
plt.show()

## Plot 2
### Pie chart of media url's domain distribution

In [ ]:
# Identify the valid URLs from the column media_urls
# Validation: This number should be equal to the number of messages with media_type as text

# def find_urls(string):
#   urls = re.findall('http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*,]|(?:%[0-9A-Fa-f][0-9A-Fa-f]))+', string)
#   return urls

# df['media_urls_cleaned'] = df.apply(lambda x: find_urls(x['media_urls']) if x['media_type'] == 'text' else [], axis=1)

# media_urls_cleaned_count = df['media_urls_cleaned'].apply(lambda x: len(x) > 0).sum()
# print(f"Number of rows with a URL in the column media_urls_cleaned: {media_urls_cleaned_count}")
url_regex = re.compile(
    r"(?:https?:\/\/)?(?:www\.)?([a-zA-Z0-9-]+\.[a-zA-Z]{2,}(?:\/\S*)?)", re.IGNORECASE
)


def find_urls(string):
    urls = url_regex.findall(string) if isinstance(string, str) else []
    return ["https://" + url if not url.startswith("http") else url for url in urls]


df["media_urls_cleaned"] = df.apply(
    lambda x: find_urls(x["media_urls"]) if x["media_type"] == "text" else [], axis=1
)
media_urls_cleaned_count = df["media_urls_cleaned"].apply(lambda x: len(x) > 0).sum()
print(
    f"Number of rows with a URL in the column media_urls_cleaned: {media_urls_cleaned_count}"
)

missing_rows = df[
    (df["media_type"] == "text") & (df["media_urls_cleaned"].apply(len) == 0)
]
print(
    f"Number of rows where there is NO URL: {len(missing_rows)}, ID's: {missing_rows[['id', 'verification_status']].values.tolist()}"
)

In [ ]:
def identify_url_source(url):
    sources = {
        "facebook": "Facebook",
        "instagram": "Instagram",
        "youtube": "YouTube",
        "x": "Twitter",
        "twitter": "Twitter",
    }

    for source, name in sources.items():
        if source in url:
            return name

    return "Other"


df["url_source"] = df.apply(
    lambda row: identify_url_source(row["media_urls_cleaned"][0])
    if len(row["media_urls_cleaned"]) > 0
    else "None",
    axis=1,
)

In [ ]:
without_none_url_source = df["url_source"].value_counts()
url_counts = without_none_url_source[without_none_url_source.index != "None"]
print(url_counts.values.sum())

In [ ]:
url_labels = url_counts.index
colors = ["#93AF7B", "#F0A7C4", "#70ACE8", "#A5CFA2", "#B4A4B3"]

plt.pie(url_counts, labels=url_labels, colors=colors, wedgeprops={"edgecolor": "black"})
# plt.title('URL Sources')
plt.show()

In [ ]:
total_counts = sum(url_counts.values)
percentages = [(count / total_counts) * 100 for count in url_counts]

plot2_df = pd.DataFrame(
    {
        "URL Source": url_labels,
        "Count": url_counts.values,
        "Percentage (%)": percentages,
    }
)
print(plot2_df)

### analyse "other" labeled url's

In [ ]:
other_urls_df = df[df["url_source"] == "Other"]
other_urls_df["media_urls"].head()

In [ ]:
def extract_domain(url_string):
    # Remove curly braces and clean the URL string
    url_string = url_string.strip("{}")
    try:
        domain = urlparse(url_string).netloc
        if not domain and url_string.startswith("http"):
            parts = url_string.split("/")
            if len(parts) > 2:
                domain = parts[2]
        return domain.lower() if domain else None
    except Exception as e:
        print(f"Error: {e}")
        return None


other_urls_df.loc[:, "domain"] = other_urls_df["media_urls_cleaned"].apply(
    lambda urls: extract_domain(urls[0]) if len(urls) > 0 else None
)

unique_domains = other_urls_df["domain"].dropna().unique()
print(f"Unique domains in 'Other' URLs: {len(unique_domains)}")
print(unique_domains)

## Plot 3
### Weekly dist of messages that hit the tipline

In [ ]:
# Group messages by week
# Group messages by week
df["week_start"] = df["inserted_at_ist"].dt.date - pd.to_timedelta(
    df["inserted_at_ist"].dt.dayofweek, unit="d"
)
df["week_end"] = df["week_start"] + pd.to_timedelta(6, unit="d")

# Count messages per week
df_grouped = (
    df.groupby(["week_start", "week_end"]).size().reset_index(name="message_count")
)

# Add these results to a dataframe and format them
df_grouped["week_start"] = pd.to_datetime(df_grouped["week_start"]).dt.date
df_grouped["week_end"] = pd.to_datetime(df_grouped["week_end"]).dt.date
df_grouped["week_start"] = df_grouped["week_start"].apply(
    lambda x: x.strftime("%B %d %Y")
)
df_grouped["week_end"] = df_grouped["week_end"].apply(lambda x: x.strftime("%B %d %Y"))

In [ ]:
df_grouped

In [ ]:
weekly_media_counts = (
    df.groupby(["week_start", "media_type"]).size().unstack(fill_value=0)
)
weekly_media_counts = weekly_media_counts.reset_index()

In [ ]:
weekly_media_counts

In [ ]:
weekly_media_counts.set_index("week_start")[["video", "audio", "text"]].plot(
    kind="bar", stacked=True, color=["#93AF7B", "#70ACE8", "#F0A7C4"], width=0.8
)
# plt.title('Media Type Distribution per Week')
plt.xlabel("Week Start Date")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## Plot 4
### Pie chart of verification labels
### missing values - 36

In [ ]:
ver_labels_counts = df["verification_status"].value_counts()
print("null values in ver status", df["verification_status"].isnull().sum())
# what id's are the null values
print(df[df["verification_status"].isnull()]["id"].tolist())

In [ ]:
# see a media item that does NOT have a verification status
df.loc[df['id'] == 3474, ['id', 'tags', 'verification_status']]

In [ ]:
ver_labels_counts_df = df["verification_status"].value_counts().reset_index()
ver_labels_counts_df.columns = ["verification_status", "count"]

In [ ]:
ver_labels_counts_df

In [ ]:
colors = [
    "#93AF7B",  # Green
    "#F0A7C4",  # Pink
    "#70ACE8",  # Blue
    "#A5CFA2",  # Light Green (interpolated)
    "#C6A7B8",  # Light Pink (interpolated)
    "#D7B2C8",  # Light Pink (interpolated)
    "#A0C4A3",  # Light Green (interpolated)
    "#BBC8AD",  # Light Green (interpolated)
    "#4D4D4D",  # Black
]

plt.pie(
    ver_labels_counts.values,
    labels=ver_labels_counts.index,
    startangle=140,
    colors=colors,
    wedgeprops={"edgecolor": "black"},
)
# plt.title('Distribution of Verification Status Labels')
plt.show()

## Plot 5
### Language and Tags Wordcloud

#### Unique Tags used in the dashboard in this timeline - 62
#### top 10 tags used

```
Real Voice - 174
Real Video - 165
Real Video, Real Voice - 158
Broken Link - 36
Trigger warning - 29
Narendra Modi - 16
Financial scams - 16
AI-video - 12
BJP - 11
Yogi Adityanath - 11
```

In [ ]:
# always update the tags list from the codebase
all_tags = [
    "Suspected cheapfake",
    "Suspected deepfake",
    "Deepfake",
    "Cheapfake",
    "AI-voice",
    "AI-video",
    "Manipulated video",
    "Manipulated audio",
    "Faceswap",
    "AI-image",
    "Real Video, AI Voice",
    "AI Video, Real Voice",
    "AI Video, AI Voice",
    "Real Video, Real Voice",
    "Real Video",
    "Trigger warning",
    "Sexual content",
    "Graphic content",
    "Profanity",
    "Out of scope",
    "Palestine Genocide",
    "Ukraine War",
    "Celeb Conduct",
    "Leader Conduct",
    "Imposter Content",
    "Islamophobia",
    "Misogyny",
    "Political Fakes",
    "Election 2024",
    "Phase-1",
    "Phase-2",
    "Phase-3",
    "Phase-4",
    "Phase-5",
    "Phase-6",
    "Phase-7",
    "Farmers' Protest",
    "Economy",
    "International Relations",
    "Pakistan",
    "China",
    "Entertainment",
    "Sports",
    "Youtube",
    "Vimeo",
    "Twitter",
    "Facebook",
    "INC",
    "BJP",
    "AAP",
    "Trinamool Congress (TMC)",
    "Bahujan Samaj Party (BSP)",
    "CPI",
    "CPIM",
    "Nationalist Congress Party (NCP)",
    "Shiv Sena",
    "Telangana Rashtra Samithi (TRS)",
    "Biju Janata Dal (BJD)",
    "Rashtriya Janata Dal (RJD)",
    "All India Trinamool Congress (TMC)",
    "Dravida Munnetra Kazhagam (DMK)",
    "Shiromani Akali Dal (SAD)",
    "United Democratic Party (Meghalaya)",
    "Naga Peoples Front",
    "Goa Forward Party",
    "Asom Gana Parishad",
    "Jharkhand Mukti Morcha",
    "YSR Congress",
    "Telugu Desam Party (TDP)",
    "AIADMK",
    "People's Democratic Party (PDP)",
    "Yuvajana Sramika Rythu Congress Party",
    "People's Party of Arunachal",
    "Janata Dal (Secular)",
    "Janata Dal (United)",
    "Bodoland Peoples Front",
    "United People's Party, Liberal",
    "Rashtriya Lok Samta Party",
    "Communist Party of India (Marxist-Leninist) Liberation",
    "Janata Congress Chhattisgarh (J)",
    "Maharashtrawadi Gomantak",
    "Indian National Lok Dal",
    "Jannayak Janta Party",
    "J&K National Conference",
    "J&K National Panthers Party",
    "J&K Peoples Democratic Party (PDP)",
    "AJSU Party",
    "Kerala Congress (M)",
    "Indian Union Muslim League",
    "Revolutionary Socialist Party",
    "Maharashtra Navnirman Seena",
    "Hill State People's Democratic Party (Meghalaya)",
    "People's Democratic Front",
    "Voice of the People Party",
    "Mizo National Front",
    "Zoram Nationalist Party",
    "Nationalist Democratic Progressive Party",
    "Lok  Janshakti Party (Ram Vilas)",
    "All India Anna Dravida Munnetra Kazhagam",
    "All India N.R. Congress (Puducherry)",
    "Rashtriya Loktantrik Party (Rajasthan)",
    "Sikkim Democratic Front",
    "Sikkim Krantikari Morcha",
    "Desiya Murpokku Dravida Kazhagam (Tamil Nadu)",
    "All India Majlis-E-Ittehadul Muslimeen (Telangana)",
    "Bharat Rashtra Samithi (Telangana)",
    "Indigenous People's Front of Tripura",
    "Tipra Motha Party",
    "Apna Dal (Soneylal) Uttar Pradesh",
    "All India Forward Bloc (West Bengal)",
    "Narendra Modi",
    "Amit Shah",
    "Rahul Gandhi",
    "Sonia Gandhi",
    "Rajnath Singh",
    "Nitin Gadkari",
    "Manoj Tiwari",
    "Piyush Goyal",
    "Yogi Adityanath",
    "JP Nadda",
    "Priyanka Gandhi",
    "Nitish Kumar",
    "Mamata Banerjee",
    "Rajnath Singh",
    "Arvind Kejriwal",
    "Rekha Gupta",
    "Mohan Bhagwat",
    "Manohar Lal Khattar",
    "Asaduddin Owaisi",
    "Kisan Mazdur Morcha (KMM)",
    "Bharat Kisan Union (BKU)",
    "Sarwan Singh Pandher",
    "Jagjit Singh Dallewal",
    "Manjeet Singh Rai",
    "Rakesh Tikait",
    "Ravish Kumar",
    "Anjana Om Kashyap",
    "Sudhir Chaudhary",
    "Punya Prasun Bajpai",
    "Rajat Sharma",
    "Arnab Goswami",
    "Rajdeep Sardesai",
    "Palki Sharma",
    "Faye D'souza",
    "EVM",
    "Virat Kohli",
    "Sachin Tendulkar",
    "RSS",
    "Aviator App",
    "Electoral Bonds",
    "DMRC",
    "Health misinformation",
    "X",
    "Instagram",
    "Financial scams",
    "Castiest",
    "Anti-minority",
    "Mahua Moitra",
    "Kanhaiya Kumar",
    "D K Shivakumar",
    "Mallikarjun Kharge",
    "B. R. Ambedkar",
    "Bheem Sena",
    "Baba Ramdev",
    "Nitin Gadkari",
    "Dhruv Rathee",
    "Amit Malviya",
    "Dinesh Lal Yadav",
    "Jamie Dimon",
    "Anant Ambani",
    "Mysore",
    "Wayanad",
    "IUML",
    "UDF",
    "AIUDF",
    "I.N.D.I.A",
    "NDA",
    "Pro INC",
    "Anti INC",
    "Pro Rahul Gandhi",
    "Anti Rahul Gandhi",
    "Pro Modi",
    "Anti Modi",
    "Pro BJP",
    "Anti BJP",
    "Reservations",
    "Inheritance Tax",
    "Caste Census",
    "Wealth Redistribution",
    "HD Kumaraswamy",
    "Dolly Sharma",
    "Annie Raja",
    "Tejasvi Surya",
    "Sam Pitroda",
    "Kuki",
    "Meitei",
    "Manipur",
    "Tom Cruise",
    "Rajat Sharma",
    "Mukesh Ambani",
    "Santosh Pandey",
    "Arun Govil",
    "Bhupesh Baghel",
    "Shashi Tharoor",
    "Hema Malini",
    "Akbaruddin Owaisi",
    "Shah Rukh Khan",
    "Akshay Kumar",
    "Anupam Kher",
    "Surat",
    "Manmohan Singh",
    "Bollywood",
    "Ranveer Singh",
    "MS Dhoni",
    "Aamir Khan",
    "Amitabh Bachchan",
    "Kangana Ranaut",
    "Language : English",
    "Language : Hindi",
    "Language : Tamil",
    "Language : Telugu",
    "Language : Urdu",
    "Language : Marathi",
    "Language : Bangla",
    "Language : Other",
    "Atishi Marlena Singh",
    "Rashmika Mandanna",
    "US Elections",
    "Exit polls",
    "Pro Arvind Kejriwal",
    "Anti Arvind Kejriwal",
    "Nandan Nilekani",
    "Palestine",
    "Israel",
    "Israel-Palestine",
    "Anti AAP",
    "Pro AAP",
    "Elon Musk",
    "Mark Zuckerberg",
    "Satire",
    "Spoof",
    "Anti India",
    "Real Voice",
    "Joe Biden",
    "Emmanuel Macron",
    "Rishi Sunak",
    "Donald Trump",
    "Kash Patel",
    "Ebrahim Raisi",
    "Akhilesh Yadav",
    "COVID-19",
    "Himachal Pradesh",
    "Andhra Pradesh",
    "Rajasthan",
    "Odisha",
    "Telangana",
    "Assam",
    "Manipur",
    "J&K",
    "Haryana",
    "Punjab",
    "Delhi",
    "Gujarat",
    "Kerala",
    "Karnataka",
    "Tamil Nadu",
    "Maharashtra",
    "Madhya Pradesh",
    "West Bengal",
    "Bihar",
    "Uttar Pradesh",
    "Alia Bhatt",
    "Ratan Tata",
    "NCSI",
    "Sonakshi Sinha",
    "Finance",
    "Nita Ambani",
    "Lip-sync",
    "Kamala Harris",
    "Jill Biden",
    "US politics",
    "Voice Clone",
    "Narayana Murthy",
    "Wion",
    "Financial Advice",
    "Hate speech",
    "Sundar Pichai",
    "Weather/Climate Hoax",
    "Broken Link",
    "Nirmala Sitharaman",
    "Shaktikanta Das",
    "Rahil Chaudhary",
    "Naresh Trehan",
    "Devi Prasad Shetty",
    "Imran Khan",
    "Sanjay Malhotra",
]

In [ ]:
print(len(all_tags))
print(len(set(all_tags)))
# 4 tags are duplicate

In [ ]:
# Find elements with "Langauge:"
langauge_tags_check = [tag for tag in all_tags if "Language : " in tag]

# Print the count and the elements
print(f"Count: {len(langauge_tags_check)}")
print("Matching elements:", langauge_tags_check)

In [ ]:
%%time
tag_count_dict = {tag: 0 for tag in all_tags}
for tag in all_tags:
    # Escape the tag to avoid regex interpretation
    escaped_tag = re.escape(tag)
    tag_count_dict[tag] = df["tags"].str.contains(escaped_tag, case=True).sum()

In [ ]:
len(tag_count_dict)

In [ ]:
sorted_tag_counts = dict(
    sorted(tag_count_dict.items(), key=lambda item: item[1], reverse=True)
)
filtered_tag_counts = {k: v for k, v in sorted_tag_counts.items() if v != 0}

In [ ]:
# total tags used in the dashboard so far
len(filtered_tag_counts)

In [ ]:
# what langauges are used
keys_to_remove = [
'Language : English', 'Language : Hindi', 'Language : Tamil', 'Language : Telugu', 'Language : Urdu', 'Language : Marathi', 'Language : Bangla', 'Language : Other'
]

remove_lang_tag_counts = {
    k: v for k, v in filtered_tag_counts.items() if k not in keys_to_remove
}
print(len(remove_lang_tag_counts))

In [ ]:
# top 10 tags used
count = 1
for key, val in remove_lang_tag_counts.items():
    print(f"{key} - {val}")
    if count == 10:
        break
    count += 1

In [ ]:
# now lets first just include the language ones
lang_tag_counts = {k: v for k, v in filtered_tag_counts.items() if k in keys_to_remove}

In [ ]:
lang_tag_counts

In [ ]:
# find id's where language is other
language_other_rows = df[
    df["tags"].str.contains("Language : Other", case=True, na=False)
]
language_other_ids = language_other_rows["id"].tolist()
print(len(language_other_ids))
print(language_other_ids)

In [ ]:
# number of items where the verification status is NOT spam or None and any language tag does not exist
filtered_df = df[
    (df["verification_status"].notnull()) &
    (df["verification_status"] != "spam")
]
lang_pattern = "|".join(re.escape(tag) for tag in keys_to_remove)
no_lang_tags_df = filtered_df[~filtered_df["tags"].str.contains(lang_pattern, na=False)]
count = no_lang_tags_df.shape[0]
print("Number of items where verification_status is not spam/None and no language tag exists:", count)

print("IDs of items with no language tag and verification_status not spam/None:")
print(no_lang_tags_df["id"].tolist())

print("Verification status distribution for these items:")
print(no_lang_tags_df["verification_status"].value_counts())

In [ ]:
# id's where
filtered_ids = no_lang_tags_df[
    no_lang_tags_df["verification_status"].isin(["ai_generated", "out_of_scope"])
]["id"]

print(filtered_ids.tolist())

In [ ]:
colors = [
    "#93AF7B",  # Green
    "#F0A7C4",  # Pink
    "#70ACE8",  # Blue
    "#4D4D4D",  # Black
]
plt.pie(
    list(lang_tag_counts.values()),
    labels=list(lang_tag_counts.keys()),
    startangle=140,
    colors=colors,
    wedgeprops={"edgecolor": "black"},
)
# plt.title('Language Dist of Media items')
plt.show()

In [ ]:
total_counts = sum(list(lang_tag_counts.values()))
percentages = [(count / total_counts) * 100 for count in list(lang_tag_counts.values())]

df_lang_counts = pd.DataFrame(
    {
        "Language Tag": list(lang_tag_counts.keys()),
        "Count": list(lang_tag_counts.values()),
        "Percentage (%)": percentages,
    }
)

In [ ]:
df_lang_counts

In [ ]:
# print all tags counts
# Print the markdown table
print("| Topic                 | Mentions |")
print("|-----------------------|----------|")

for key, value in remove_lang_tag_counts.items():
    print(f"| {key:23} | {value:8} |")